# Railtracks Flag Analysis

After the GNN (or RF) fraud-detection process, this notebook loads the **list of flagged items** and uses a **Railtracks** agent to analyze **why** they were flagged and **what red flags / signs** indicate potential fraud.

**Inputs:**
- Path to a **flags JSON** file (e.g. `outputs/gnn_flagged_accounts_<ts>.json` from the GNN notebook), or
- Path to an **RF predictions CSV** (e.g. `outputs/rf_eval_predictions_<ts>.csv`); we filter to high-risk rows and convert to the same format.

**Output:** Printed analysis + optional report saved to `outputs/flag_analysis_report_<timestamp>.md`.

**LLM:** Uses **Gemini free tier** by default. Put keys in a `.env` file in the project root (e.g. `GEMINI_API_KEY=...`) or set them in your environment. Get a free Gemini key at [Google AI Studio](https://aistudio.google.com). Use `OPENAI_API_KEY` in `.env` to use OpenAI instead.

## 1. Setup

In [ ]:
# Install if needed (uncomment in Colab or fresh env)
# %pip install railtracks

import json
import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()  # load GEMINI_API_KEY / OPENAI_API_KEY from .env in project root

import pandas as pd
import railtracks as rt

# Keys: put in .env (e.g. GEMINI_API_KEY=..., OPENAI_API_KEY=...) or set in environment.
# Gemini free: https://aistudio.google.com

## 2. Load flags

In [ ]:
# Option A: path to flags JSON (from GNN notebook export)
FLAGS_JSON_PATH = None  # e.g. Path("outputs/gnn_flagged_accounts_20260314_120000.json")

# Option B: path to RF predictions CSV (we filter to pred_label==1 or high pred_probability)
RF_PREDICTIONS_CSV_PATH = Path("outputs/rf_eval_predictions_20260314_003135.csv")  # or None

MAX_FLAGS = 50  # cap number of flags sent to the LLM to keep prompt size bounded
RF_PROB_THRESHOLD = 0.0  # min pred_probability to count as flag when using RF CSV (0 = take top by score)

def load_flags_from_json(path: Path) -> list:
    with open(path, "r") as f:
        return json.load(f)

def load_flags_from_rf_csv(path: Path, prob_threshold: float = 0.0, max_flags: int = 100) -> list:
    df = pd.read_csv(path)
    flagged = df.nlargest(max_flags, "pred_probability")
    if prob_threshold > 0:
        flagged = flagged[flagged["pred_probability"] >= prob_threshold]
    feature_cols = [c for c in flagged.columns if c not in ("actual_label", "pred_label", "pred_probability")]
    flags = []
    for idx, row in flagged.iterrows():
        features = {str(c): (int(v) if isinstance(v, (int, float)) and float(v) == int(v) else v) for c, v in row[feature_cols].items()}
        flags.append({
            "id": f"tx_{idx}",
            "risk_score": float(row["pred_probability"]),
            "source": "rf",
            "features": features,
        })
    return flags

if FLAGS_JSON_PATH and Path(FLAGS_JSON_PATH).exists():
    flags = load_flags_from_json(Path(FLAGS_JSON_PATH))
    print(f"Loaded {len(flags)} flags from JSON.")
elif RF_PREDICTIONS_CSV_PATH and RF_PREDICTIONS_CSV_PATH.exists():
    flags = load_flags_from_rf_csv(RF_PREDICTIONS_CSV_PATH, prob_threshold=RF_PROB_THRESHOLD, max_flags=MAX_FLAGS)
    print(f"Loaded {len(flags)} flags from RF CSV (prob >= {RF_PROB_THRESHOLD}).")
else:
    flags = []
    print("No valid FLAGS_JSON_PATH or RF_PREDICTIONS_CSV_PATH. Set one and ensure the file exists.")

flags = sorted(flags, key=lambda x: -x["risk_score"])[:MAX_FLAGS]

## 3. Format flags for the agent

In [ ]:
def format_flags_for_prompt(flags: list) -> str:
    lines = []
    for i, f in enumerate(flags, 1):
        lines.append(f"--- Flag {i} ---")
        lines.append(f"  id: {f.get('id', 'N/A')}")
        lines.append(f"  risk_score: {f.get('risk_score', 0):.4f}")
        lines.append(f"  source: {f.get('source', 'N/A')}")
        if f.get("features"):
            for k, v in f["features"].items():
                lines.append(f"  {k}: {v}")
        lines.append("")
    return "\n".join(lines) if lines else "(No flags provided.)"

formatted = format_flags_for_prompt(flags)
print(f"Formatted {len(flags)} flags. Preview:\n")
print(formatted[:2000] + ("..." if len(formatted) > 2000 else ""))

## 4. Railtracks agent: analyze why flagged and what are the signs

In [ ]:
SYSTEM_MESSAGE = """You are an expert AML (Anti-Money Laundering) analyst. You are given a list of transactions or accounts that have been flagged by a fraud detection system (GNN or Random Forest). For each flagged item, the list includes an id, a risk score, and various features (amounts, currencies, payment format, time of day, etc.).

Your task:
1. Explain WHY each flagged item (or groups of similar items) is considered high risk.
2. Summarize the RED FLAGS and SIGNS that indicate potential fraud or money laundering (e.g. unusual amounts, velocity, currency patterns, time patterns, structural cues).
3. If the list is long, focus on the highest-risk items and on common patterns across the set.

Be concise but specific. Use the feature names and values from the list in your explanation."""

user_message = (
    "Below is the list of flagged items from our fraud detection system. "
    "Analyze why they are flagged and what signs indicate potential fraud.\n\n"
    + formatted
)

# Use Gemini free tier if GEMINI_API_KEY is set (Railtracks uses LiteLLM; gemini/ prefix = Google AI Studio)
if os.environ.get("GEMINI_API_KEY"):
    import litellm
    response = litellm.completion(
        model="gemini-3-flash-preview",
        messages=[
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": user_message},
        ],
    )
    report_text = response.choices[0].message.content or ""
    print("Using Gemini (free tier).")
else:
    agent = rt.agent_node(
        "AML Analyst",
        tool_nodes=(),
        llm=rt.llm.OpenAILLM("gpt-4o"),
        system_message=SYSTEM_MESSAGE,
    )
    flow = rt.Flow(name="Flag Analysis", entry_point=agent)
    result = flow.invoke(user_message)
    report_text = result.text if hasattr(result, "text") else str(result)

## 5. Report

In [ ]:
print(report_text)

# Save report to outputs/
from datetime import datetime
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
report_path = OUTPUT_DIR / f"flag_analysis_report_{ts}.md"
with open(report_path, "w") as f:
    f.write("# Flag Analysis Report\n\n")
    f.write(report_text)
print(f"\nReport saved to: {report_path}")